# PN2N Colab Notebook

**Poisson-thinning-enabled self-supervised denoising for photon-limited microscopy**

**Author:** Hongqiang Ma, PhD  
**Affiliation:** Department of Bioengineering and Beckman Institute for Advanced Science and Technology, University of Illinois Urbana-Champaign  
**Contact:** mhq@illinois.edu  
**Notebook purpose:** This Google Colab notebook trains and applies PN2N to 2D microscopy images. PN2N generates statistically independent self-supervised training pairs from a single photon-counted image using Poisson thinning, then applies the trained network to the original full-photon input for denoising.

**Associated manuscript:** *Poisson thinning enables self-supervised denoising in photon-limited microscopy*.

**Expected input:** one or more `.tif`/`.tiff` images in a Google Drive folder. Images should be raw detector images or photon-calibrated images. If the input is in digital numbers, set the camera offset and gain below so the code can convert the data to photon units before Poisson thinning.

**Main output:** denoised TIFF files saved with the suffix `_PN2N`, together with model weights and a CSV loss log for each processed image.

**Recommended citation:** Please cite the associated manuscript if this notebook or PN2N is useful for your work.


## Workflow

This notebook follows the same logic as the PN2N method:

1. Mount Google Drive and load TIFF images.
2. Convert raw camera values to photon units using camera offset and gain.
3. Randomly crop training patches.
4. Generate two Poisson-thinned views using binomial splitting.
5. Train a U-Net in a bidirectional Noise2Noise-style manner: view 1 → view 2 and view 2 → view 1.
6. Apply the trained model to the original full-photon image or stack.
7. Save denoised TIFF outputs, model weights and training logs.

For a quick code test, reduce the number of epochs. For manuscript-quality results, use the training settings reported in the paper or supplementary methods.


## 1. Imports and Google Drive setup

In [8]:
# Core Python and scientific-computing libraries
import os
import time
# PyTorch is used for U-Net training and GPU acceleration
import torch
# PyTorch is used for U-Net training and GPU acceleration
import torch.nn as nn
# PyTorch is used for U-Net training and GPU acceleration
import torch.nn.functional as F
import numpy as np
# tifffile is used to read and write microscopy TIFF images/stacks
import tifffile
import csv  # Added for CSV export
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from torch.cuda.amp import autocast, GradScaler
# Google Drive is used as the input/output file system in Colab
from google.colab import drive

# Mount Google Drive. Change the paths in the parameter cell below to your own folders.
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 2. Helper functions

In [9]:
# --- 1. Helper functions ---
# These utilities provide reproducibility, image loading and edge handling.
def setup_seed(seed):
    """Set random seeds for reproducible training and patch sampling."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = True

def load_2D_data(data_folder):
    """Load all 2D TIFF images or all frames from 3D TIFF stacks in a folder.

    Note: this helper is kept for compatibility with earlier versions.
    The batch workflow below trains on one image/stack at a time using
    load_single_image_as_training_stack().
    """
    if not os.path.exists(data_folder):
        raise FileNotFoundError(f"Input path not found: {data_folder}")

    train_data = []
    files = [f for f in os.listdir(data_folder) if f.lower().endswith(('.tif', '.tiff'))]

    if not files:
        print(f"Warning: No .tif/.tiff files found in {data_folder}")
        return np.array([])

    for filename in files:
        filepath = os.path.join(data_folder, filename)
        img = tifffile.imread(filepath)
        if img.ndim == 2:
            train_data.append(img)
        elif img.ndim == 3:
            train_data.extend(img[i] for i in range(img.shape[0]))

    if not train_data:
        return np.array([])

    return np.stack(train_data, axis=0)

def pad_edge_batch(img_batch_tensor, pad_size=16):
    """Apply reflection padding before inference/training to reduce edge artifacts."""
    if img_batch_tensor.dim() != 4:
        raise ValueError("Input tensor must be 4D (B, C, H, W)")
    padding = (pad_size, pad_size, pad_size, pad_size)
    return F.pad(img_batch_tensor, padding, mode='reflect')

def crop_edge_batch(img_batch_tensor, crop_size=16):
    """Remove the padding region after the network prediction."""
    if img_batch_tensor.dim() != 4:
        raise ValueError("Input tensor must be 4D (B, C, H, W)")
    _, _, H, W = img_batch_tensor.shape
    return img_batch_tensor[:, :, crop_size:-crop_size, crop_size:-crop_size]

## 3. U-Net denoising model

In [10]:
# --- 2. U-Net model architecture ---
# A compact 2D U-Net is used as the PN2N denoising network.
# The network maps one Poisson-thinned view to the other during training,
# and is later applied to the original full-photon image for inference.
class UNet2D(nn.Module):
    """2D U-Net denoising network used by PN2N.

    Parameters
    ----------
    init_channels : int
        Number of feature channels in the first encoder level.
    num_layer : int
        Depth of the encoder-decoder network.
    """
    def __init__(self, init_channels = 32, num_layer=4):
        assert num_layer > 2, 'Number of layers can not be smaller than 3.'
        super(UNet2D, self).__init__()

        encoder = []
        encoder.append(UNet2D_Block_First(1,init_channels))
        for i in range(num_layer-2):
            encoder.append(UNet2D_Encoder_Block(init_channels*(2**i), init_channels*(2**(i+1)) ))
        self.encoder = nn.Sequential(*encoder)

        self.center = UNet2D_Center_Block(init_channels*(2**(num_layer-2)),
                                          init_channels*(2**(num_layer-1)),
                                          init_channels*(2**(num_layer-2)))

        decoder = []
        for i in reversed(range(num_layer-2)):
            decoder.append(UNet2D_Decoder_Block(init_channels*(2**(i+1)), init_channels*(2**i)))
        decoder.append(UNet2D_Block_Last(init_channels,1))
        self.decoder = nn.Sequential(*decoder)

    def forward(self, x):
        # Store encoder outputs for skip connections.
        encoder_out = [x]
        for encoder_layer in self.encoder:
            encoder_out.append(encoder_layer(encoder_out[-1]))

        decoder_out = [self.center(encoder_out[-1])]
        for idx, decoder_layer in enumerate(self.decoder):
            encoder_out_idx = encoder_out[-1-idx]
            decoder_out_idx = zero_padding(decoder_out[-1], encoder_out_idx.shape)
            # Concatenate decoder features with the corresponding encoder features.
            concat_in = torch.cat([decoder_out_idx, encoder_out_idx], dim=1)
            decoder_out.append(decoder_layer(concat_in))

        return decoder_out[-1]

class UNet2D_Block_First(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet2D_Block_First, self).__init__()
        block = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        ]
        self.block = nn.Sequential(*block)
    def forward(self, x): return self.block(x)

class UNet2D_Encoder_Block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet2D_Encoder_Block, self).__init__()
        block = [
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        ]
        self.block = nn.Sequential(*block)
    def forward(self, x): return self.block(x)

class UNet2D_Center_Block(nn.Module):
    def __init__(self, in_channels, middle_channels, out_channels):
        super(UNet2D_Center_Block, self).__init__()
        block = [
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels, middle_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(middle_channels, middle_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(middle_channels, out_channels, kernel_size=2, stride=2)
        ]
        self.block = nn.Sequential(*block)
    def forward(self, x): return self.block(x)

class UNet2D_Decoder_Block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet2D_Decoder_Block, self).__init__()
        block = [
            nn.Conv2d(2*in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        ]
        self.block = nn.Sequential(*block)
    def forward(self, x): return self.block(x)

class UNet2D_Block_Last(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet2D_Block_Last, self).__init__()
        block = [
            nn.Conv2d(2*in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
        ]
        self.block = nn.Sequential(*block)
    def forward(self, x): return self.block(x)

def zero_padding(this, shp):
    """Pad decoder features so their size matches the skip-connection tensor."""
    if len(shp) == 4:
        pad = (0, shp[3] - this.shape[3], 0, shp[2] - this.shape[2])
    elif len(shp) == 5:
        pad = (0, shp[4] - this.shape[4], 0, shp[3] - this.shape[3], 0, shp[2] - this.shape[2])
    return F.pad(this, pad)

## 4. Poisson-thinning dataset and normalization

In [11]:
# --- 3. PN2N dataset and pair normalization ---
# Each training sample is a random patch from the input image/stack.
# The patch is split by binomial random sampling into two independent
# Poisson-thinned views that share the same underlying structure.
class unet_2D_dataset(Dataset):
    """Random-patch dataset for PN2N self-supervised training.

    For each sampled patch, photon counts are split as:
        patch_bi1 ~ Binomial(patch, 0.5)
        patch_bi2 = patch - patch_bi1

    If the original patch is Poisson distributed, these two views are
    conditionally independent Poisson observations of the same signal.
    """
    def __init__(self, data, num_sample, crop_size=128):
        self.crop_size = crop_size
        self.num_sample = num_sample
        self.data = data

    def __getitem__(self, index):
        N, H, W = self.data.shape
        crop_size = self.crop_size

        # Randomly choose one frame/slice and one spatial crop.
        random_n = np.random.randint(0, N)
        start_h = np.random.randint(0, H - crop_size + 1)
        start_w = np.random.randint(0, W - crop_size + 1)

        patch = self.data[random_n, start_h:start_h+crop_size, start_w:start_w+crop_size]

        # Poisson thinning: binomially split each photon count into two views.
        # Here p = 0.5 gives two views with matched expected photon counts.
        patch_bi1 = np.random.binomial(patch, 0.5)
        patch_bi2 = patch - patch_bi1

        img_1, img_2 = mean_std_norm_pair(patch_bi1, patch_bi2)

        img_1 = torch.from_numpy(img_1).unsqueeze(0).float()
        img_2 = torch.from_numpy(img_2).unsqueeze(0).float()

        return img_1, img_2

    def __len__(self):
        return self.num_sample

def mean_std_norm_pair(img1, img2):
    """Jointly normalize a thinned image pair using shared mean and std.

    Joint normalization keeps the two views on the same intensity scale and
    avoids introducing an artificial scale mismatch between input and target.
    """
    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)
    mean_v = np.mean([img1,img2])
    std_v = np.std([img1,img2])
    img1_norm = (img1-mean_v)/(std_v+1e-7)
    img2_norm = (img2-mean_v)/(std_v+1e-7)
    return img1_norm, img2_norm

## 5. PN2N training loop

In [12]:
# --- 4. PN2N training loop ---
# The network is trained bidirectionally: view 1 predicts view 2 and view 2
# predicts view 1. Because the two views have independent shot noise, the
# shared signal is reinforced while uncorrelated noise is not.
def train_fun(model, epochs, lr, train_loader, save_dir, log_file, noisy_example, camera_gain, camera_offset, mean_v, std_v, device):
    """Train a PN2N model for one image or image stack.

    The model weights, best checkpoint and loss log are saved in save_dir.
    noisy_example, camera_gain and camera_offset are retained in the function
    signature for compatibility with earlier versions of the notebook.
    """
    start_time = time.time()
    # AdamW provides stable optimization for the compact U-Net.
    # fused=True accelerates training on supported CUDA runtimes in Colab.
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, fused=True)
    loss_fn = nn.L1Loss()
    # Automatic mixed precision reduces GPU memory use and improves speed.
    scaler = torch.amp.GradScaler('cuda')

    # --- Early Stopping & Smoothing Parameters ---
    patience = 1000           # Stop if no improvement for 500 epochs
    min_delta = 0.001      # Minimum change to count as improvement
    best_loss = float('inf')
    counter = 0
    smooth_loss = None
    smoothing_factor = 0.9  # How much to weight previous values (0.9 = smooth)

    # Setup CSV logging
    csv_path = os.path.join(save_dir, 'loss_log.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'loss', 'smooth_loss', 'elapsed_time'])

    pbar = tqdm(range(epochs), desc="Training PN2N", unit="epoch")

    for epoch in pbar:
        model.train()
        train_loss_batch = []

        for i, sample in enumerate(train_loader):
            optimizer.zero_grad()
            image_1 = pad_edge_batch(sample[0]).to(device, non_blocking=True, memory_format=torch.channels_last)
            image_2 = pad_edge_batch(sample[1]).to(device, non_blocking=True, memory_format=torch.channels_last)

            # Bidirectional PN2N training: concatenate both directions in one batch.
            input_tensor = torch.vstack([image_1, image_2])
            target_tensor = torch.vstack([image_2, image_1])

            with torch.amp.autocast('cuda'):
                output_tensor = model(input_tensor)
                # Compare only the unpadded central region.
                loss = loss_fn(crop_edge_batch(output_tensor), crop_edge_batch(target_tensor))

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss_batch.append(loss.item())

        # Calculate current epoch loss
        train_loss_epoch = np.mean(train_loss_batch)

        # --- Apply Smoothing (Exponential Moving Average) ---
        if smooth_loss is None:
            smooth_loss = train_loss_epoch
        else:
            smooth_loss = (smoothing_factor * smooth_loss) + ((1 - smoothing_factor) * train_loss_epoch)

        elapsed = time.strftime('%H:%M:%S', time.gmtime(int(time.time() - start_time)))

        # Log to CSV (including smooth loss)
        with open(csv_path, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, f"{train_loss_epoch:.6f}", f"{smooth_loss:.6f}", elapsed])

        pbar.set_postfix({'Loss': f'{train_loss_epoch:.4f}', 'Smooth': f'{smooth_loss:.4f}'})

        # --- Early Stopping Logic ---
        if smooth_loss < (best_loss - min_delta):
            best_loss = smooth_loss
            counter = 0
            # Save the best model so far
            torch.save(model.state_dict(), os.path.join(save_dir, 'PN2N_2D_best.pt'))
        else:
            counter += 1
            if counter >= patience:
                print(f"\nEarly stopping triggered at epoch {epoch + 1}. No improvement in smoothed loss for {patience} epochs.")
                break

    # Save final weights at the end of training or after early stopping.
    torch.save(model.state_dict(), os.path.join(save_dir, 'PN2N_2D_final.pt'))

## 6. Inference and batch-processing helpers

In [13]:
# --- 5. Batch-processing helpers and inference ---
# These functions list TIFF files, prepare each file as a training stack,
# run inference and save the denoised result.
def list_image_files(data_dir):
    """Return all TIFF files in the input folder."""
    return sorted([
        f for f in os.listdir(data_dir)
        if f.lower().endswith(('.tif', '.tiff'))
    ])

def add_suffix_to_filename(filename, suffix='_PN2N'):
    """Convert sample.tif -> sample_PN2N.tif."""
    base, ext = os.path.splitext(filename)
    return f"{base}{suffix}{ext}"

def load_single_image_as_training_stack(image_path):
    """
    Load one image file for training.
    2D TIFF -> shape (1, H, W)
    3D TIFF stack -> shape (N, H, W), where N is the number of frames/slices
    """
    img = tifffile.imread(image_path)
    if img.ndim == 2:
        return img[np.newaxis, :, :]
    elif img.ndim == 3:
        return img
    else:
        raise ValueError(f"Unsupported image dimension {img.ndim} for {image_path}. Expected 2D or 3D TIFF.")

def infer_single_file(model, image_path, save_path, camera_gain, camera_offset, mean_v, std_v, device):
    """Denoise one 2D TIFF or one 3D TIFF stack and save it to save_path."""
    model.eval()
    img_i = tifffile.imread(image_path)

    def denoise_2d_frame(frame):
        """Denoise one 2D frame using the trained PN2N model."""
        with torch.no_grad():
            noisy_input = frame.astype(np.float32).copy()
            # Convert raw detector units to photon units. Values below the
            # camera offset are clipped before gain correction.
            noisy_input[noisy_input < camera_offset] = camera_offset
            noisy_input = (noisy_input - camera_offset) / camera_gain
            # Normalize using the dataset-level statistics from training.
            noisy_input = (noisy_input - mean_v) / (std_v + 1e-7)
            noisy_input = torch.tensor(noisy_input, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            noisy_input = pad_edge_batch(noisy_input)
            denoised_image = model(noisy_input.to(device))
            denoised_image = crop_edge_batch(denoised_image)
            denoised_image = denoised_image.detach().cpu().numpy()[0, 0]
            # Transform the network output back to the original detector scale.
            denoised_image = denoised_image * (std_v + 1e-7) + mean_v
            denoised_image = denoised_image * camera_gain + camera_offset
            denoised_image[denoised_image < 0] = 0
            return denoised_image.astype(np.float32)

    if img_i.ndim == 2:
        denoised = denoise_2d_frame(img_i)
    elif img_i.ndim == 3:
        denoised = np.stack([denoise_2d_frame(img_i[i]) for i in range(img_i.shape[0])], axis=0)
    else:
        raise ValueError(f"Unsupported image dimension {img_i.ndim} for {image_path}. Expected 2D or 3D TIFF.")

    tifffile.imwrite(save_path, denoised.astype(np.float32))

## 7. User parameters and batch execution

Set `data_dir`, `save_dir`, camera calibration and training parameters here, then run this cell to train and denoise each TIFF file independently.

In [14]:
# --- 6. User parameters and main execution ---
# Edit the Colab form fields below before running the notebook.

#@markdown # **2D Denoising Parameters**
#@markdown > Path to the noisy images
data_dir = '/content/drive/MyDrive/ColabNotebooks/images' #@param {type:"string"}  # Folder containing input .tif/.tiff files
#@markdown > Path to save results
save_dir = '/content/drive/MyDrive/ColabNotebooks/results' #@param {type:"string"}  # Folder for PN2N outputs, weights and logs

#@markdown ---
#@markdown ## Training Settings
#@markdown > Learning rate
lr = 1e-3 #@param {type:"number"}
#@markdown > Batch Size
batch_size = 32  #@param {type:"integer"}
#@markdown > Image patch size
patch_size = 256  #@param {type:"integer"}
#@markdown > Maximum training epochs
epochs = 1000  #@param {type:"integer"}
#@markdown > Random seed
seed = 42  #@param {type:"integer"}

#@markdown ---
#@markdown ## Hardware/Camera Calibration
#@markdown > Camera offset (black level)
camera_offset = 0  #@param {type:"number"}  # Camera black level; use 0 if data are already offset-corrected
#@markdown > Camera gain
camera_gain = 1  #@param {type:"number"}  # DN-to-photon conversion gain; use 1 if data are already in photon units

log_file = 'pn2n_log'

if __name__ == "__main__":
    print("PN2N batch denoising start...")
    setup_seed(seed)

    if not os.path.exists(data_dir):
        print(f"ERROR: Input directory not found: {data_dir}")
    else:
        os.makedirs(save_dir, exist_ok=True)
        model_root = os.path.join(save_dir, 'PN2N_models_and_logs')
        os.makedirs(model_root, exist_ok=True)

        # Use GPU if Colab provides one. Runtime > Change runtime type > GPU.
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")

        files = list_image_files(data_dir)
        if len(files) == 0:
            print(f"ERROR: No .tif/.tiff files found in {data_dir}")
        else:
            print(f"Found {len(files)} image file(s). Each file will be trained and denoised independently.")

            for file_idx, filename in enumerate(files, start=1):
                input_path = os.path.join(data_dir, filename)
                output_name = add_suffix_to_filename(filename, suffix='_PN2N')
                output_path = os.path.join(save_dir, output_name)

                image_base = os.path.splitext(filename)[0]
                image_work_dir = os.path.join(model_root, image_base)
                os.makedirs(image_work_dir, exist_ok=True)

                print("\n" + "=" * 80)
                print(f"[{file_idx}/{len(files)}] Processing: {filename}")
                print(f"Output will be saved as: {output_name}")

                # Load only this image/stack for this training run
                image_data = load_single_image_as_training_stack(input_path)
                image_data = image_data.astype(np.float32)
                # Convert input image from detector digital numbers to photon counts.
                # This step is essential because PN2N thinning assumes integer photon counts.
                image_data[image_data < camera_offset] = camera_offset
                image_data = (image_data - camera_offset) / camera_gain
                # Round to non-negative integer counts before binomial splitting.
                image_data = np.round(image_data).astype(np.uint16)

                # Dataset-level normalization statistics used for both training and inference.
                mean_v = np.mean(image_data)
                std_v = np.std(image_data)
                if std_v < 1e-7:
                    print(f"WARNING: {filename} has near-zero standard deviation. Skipping this file.")
                    continue

                # Approximate number of random patches sampled per epoch.
                num_sample = max(1, image_data.size // (patch_size ** 2))
                print(f"Training data shape: {image_data.shape}, Samples per epoch: {num_sample}")

                noisy_example = image_data[0].copy().astype(np.float32)
                noisy_example = (noisy_example - mean_v) / (std_v + 1e-7)
                noisy_example = torch.tensor(noisy_example, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

                train_dataset = unet_2D_dataset(image_data, num_sample, patch_size)
                train_loader = DataLoader(
                    train_dataset,
                    batch_size=batch_size,
                    shuffle=True,
                    num_workers=2,
                    pin_memory=True,
                    persistent_workers=True
                )

                # Fresh model for each image, so each output is trained only from that image
                model = UNet2D().to(device, memory_format=torch.channels_last)

                try:
                    print("Compiling model...")
                    model = torch.compile(model)
                except Exception as e:
                    print(f"Compile skipped: {e}")

                train_fun(
                    model, epochs, lr, train_loader, image_work_dir, log_file,
                    noisy_example, camera_gain, camera_offset, mean_v, std_v, device
                )

                # Apply the trained model to the original full-photon image/stack.
                infer_single_file(
                    model, input_path, output_path,
                    camera_gain, camera_offset, mean_v, std_v, device
                )
                print(f"Saved: {output_path}")

            print("\nBatch PN2N denoising completed.")

PN2N batch denoising start...
Using device: cuda:0
Found 2 image file(s). Each file will be trained and denoised independently.

[1/2] Processing: example_simulation_1.tif
Output will be saved as: example_simulation_1_PN2N.tif
Training data shape: (1, 1024, 1024), Samples per epoch: 16
Compiling model...


Training PN2N:   0%|          | 0/1000 [00:00<?, ?epoch/s]

Saved: /content/drive/MyDrive/ColabNotebooks/results/example_simulation_1_PN2N.tif

[2/2] Processing: example_simulation_2.tif
Output will be saved as: example_simulation_2_PN2N.tif
Training data shape: (1, 1024, 1024), Samples per epoch: 16
Compiling model...


Training PN2N:   0%|          | 0/1000 [00:00<?, ?epoch/s]

Saved: /content/drive/MyDrive/ColabNotebooks/results/example_simulation_2_PN2N.tif

Batch PN2N denoising completed.
